# Chapter 15
## Canard Explosions
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter15.ipynb)

## About this chapter

Near a subcritical Hopf point, the limit-cycle amplitude can grow from a
small oscillation to a full relaxation-oscillation spike within an
exponentially small window of the bifurcation parameter -- a "canard
explosion." The FitzHugh-Nagumo model and a reduced HH model both show
this; the last example (mixed-mode oscillations) adds slow spike-triggered
adaptation to produce irregular small/large oscillation patterns.

$$
\dot v=v-v^3/3-n+I,\qquad \dot n=(av-n)/\tau_n.
$$

See [`README.md`](chapter15.md) for the full guide, including suggested
order and related chapters. **Note:** several cells below reproduce
computationally expensive figures (the canard explosion needs ~1e6-step
trajectories bisected over many candidate currents) -- expect them to take
tens of seconds to minutes.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact

## Canard Explosion: FitzHugh-Nagumo Limit-Cycle Amplitude

Bisects on $I$ to find the current giving each target limit-cycle
amplitude -- the explosive growth happens in a tiny window of $I$.

In [ ]:
def simulate_canard(a=5.0, tau_n=60.0, t_final=10000.0, dt=0.01, amp_vec=(1, 2, 3, 3.5, 3.78)):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tail_start = 4 * m_steps // 5

    def amplitude(I):
        """Heun/RK2 integration using plain floats (no array storage) -- just
        the max-min of v over the tail 1/5 of the run, needed by the bisection
        below. Plain-float arithmetic here instead of numpy scalars/arrays is
        what keeps ~100 of these (1e6 steps each) practical to run."""
        v, n = 0.0, 0.1
        v_max, v_min = -np.inf, np.inf
        for k in range(m_steps):
            v_inc = v - v ** 3 / 3 - n + I
            n_inc = (a * v - n) / tau_n
            v_tmp = v + dt05 * v_inc
            n_tmp = n + dt05 * n_inc
            v_inc = v_tmp - v_tmp ** 3 / 3 - n_tmp + I
            n_inc = (a * v_tmp - n_tmp) / tau_n
            v = v + dt * v_inc
            n = n + dt * n_inc
            if k >= tail_start:
                if v > v_max:
                    v_max = v
                if v < v_min:
                    v_min = v
        return v_max - v_min

    def simulate(I):
        """same integration, but returns the full (v, n) trajectories"""
        v = np.zeros(m_steps + 1)
        n = np.zeros(m_steps + 1)
        v[0], n[0] = 0.0, 0.1
        for k in range(m_steps):
            v_inc = v[k] - v[k] ** 3 / 3 - n[k] + I
            n_inc = (a * v[k] - n[k]) / tau_n
            v_tmp = v[k] + dt05 * v_inc
            n_tmp = n[k] + dt05 * n_inc
            v_inc = v_tmp - v_tmp ** 3 / 3 - n_tmp + I
            n_inc = (a * v_tmp - n_tmp) / tau_n
            v[k + 1] = v[k] + dt * v_inc
            n[k + 1] = n[k] + dt * n_inc
        return v, n

    def find_I_for_amplitude(amp_target, i_left=-4.28, i_right=-4.25, tol=1e-10):
        """bisect on I so the limit-cycle amplitude (max-min of v, tail 1/5 of
        the run) matches amp_target -- the canard explosion is so steep that
        the whole amplitude range 1..3.78 lives in a tiny window of I"""
        while i_right - i_left > tol:
            I = (i_left + i_right) / 2
            amp = amplitude(I)
            if amp > amp_target:
                i_right = I
            else:
                i_left = I
        I = (i_left + i_right) / 2
        return I, simulate(I)

    results = []
    for amp_target in amp_vec:
        I, (v, n) = find_I_for_amplitude(amp_target)
        results.append((amp_target, I, v, n))
    return results, m_steps, dt, a


def plot_canard(results, m_steps, dt, a):
    fig = plt.figure(figsize=(10, 8))
    ax_phase = fig.add_subplot(2, 2, 1)
    ax_t = {1: fig.add_subplot(2, 2, 2), 2: fig.add_subplot(2, 2, 3),
            5: fig.add_subplot(2, 2, 4)}

    A, B, C, D = -3, 2, -6, -2
    L_200 = round(200 / dt)

    for ijk, (amp_target, I, v, n) in enumerate(results, start=1):
        half = v[m_steps // 2:]
        n_half = n[m_steps // 2:]
        if amp_target == 3.5:
            ax_phase.plot(half, n_half, '-b', linewidth=3)
        else:
            ax_phase.plot(half, n_half, '-k', linewidth=1)

        if ijk in ax_t:
            t = np.arange(L_200 + 1) * dt
            ax_t[ijk].plot(t, half[-(L_200 + 1):], '-k', linewidth=2)
            ax_t[ijk].set_xlabel('$t$')
            ax_t[ijk].set_ylabel('$v$')
            ax_t[ijk].set_title(f'$I={I:.7g}$')
            ax_t[ijk].set_xlim(0, 200)
            ax_t[ijk].set_ylim(-3, 2)

    v_line = A + np.arange(101) / 100 * (B - A)
    I_last = results[-1][1]
    ax_phase.plot(v_line, v_line - v_line ** 3 / 3 + I_last, '-g', linewidth=2)
    ax_phase.plot(v_line, a * v_line, '-r', linewidth=2)
    ax_phase.set_xlim(A, B)
    ax_phase.set_ylim(C, D)
    ax_phase.set_box_aspect(1)
    ax_phase.set_xlabel('$v$')
    ax_phase.set_ylabel('$n$')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_canard(*simulate_canard())

## Canard Explosion: a Single Canard Trajectory

Integrates once at a specific $I$ chosen to sit right in the canard
explosion window, showing the characteristic "duck"-shaped trajectory
that hugs the unstable middle branch of the cubic nullcline.

In [ ]:
def simulate_canard_2(a=5.0, tau_n=60.0, I=-4.256889, t_final=10000.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    v = np.zeros(m_steps + 1)
    n = np.zeros(m_steps + 1)
    v[0], n[0] = 0.0, 0.0

    for k in range(m_steps):
        v_inc = v[k] - v[k] ** 3 / 3 - n[k] + I
        n_inc = (a * v[k] - n[k]) / tau_n
        v_tmp = v[k] + dt05 * v_inc
        n_tmp = n[k] + dt05 * n_inc
        v_inc = v_tmp - v_tmp ** 3 / 3 - n_tmp + I
        n_inc = (a * v_tmp - n_tmp) / tau_n
        v[k + 1] = v[k] + dt * v_inc
        n[k + 1] = n[k] + dt * n_inc

    tail_start = 4 * m_steps // 5 - 1  # matlab's v(4*m_steps/5 : m_steps+1) is 1-indexed
    vv = v[tail_start:]
    nn = n[tail_start:]
    return vv, nn, a, I


def plot_canard_2(vv, nn, a, I):
    plt.figure(figsize=(6, 6))
    plt.plot(vv, nn, '-b', linewidth=3)

    A, B, C, D = -2, 0.5, -5, -4
    v_plot = A + np.arange(101) / 100 * (B - A)
    plt.plot(v_plot, v_plot - v_plot ** 3 / 3 + I, '-g', linewidth=2)
    plt.plot(v_plot, a * v_plot, '-r', linewidth=2)

    plt.xlim(A, B)
    plt.ylim(C, D)
    plt.gca().set_box_aspect(1)
    plt.xlabel('$v$')
    plt.ylabel('$n$')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_canard_2(*simulate_canard_2())

## FitzHugh-Nagumo Bifurcation Diagram (Macro View)

Fixed-point branches classified by Jacobian eigenvalues, plus the
limit-cycle envelope (max/min of $v$) swept over $I$.

In [ ]:
def simulate_fitzhugh_nagumo_macro(a=5.0, tau_n=60.0, i_ext_vec=None,
                                    t_final=200.0, dt=0.01, i_scan=None):
    if i_ext_vec is None:
        i_ext_vec = -6 + np.arange(501) / 500 * 12
    if i_scan is None:
        i_scan = np.arange(-5, 5 + 1e-9, 0.05)

    def f(v, i_ext):
        """zeros of this function are fixed points of the FN system"""
        return v - v ** 3 / 3 - a * v + i_ext

    def find_fixed_point(i_ext):
        v_left, v_right = -1.0, 1.0
        while f(v_left, i_ext) < 0:
            v_left -= 1
        while f(v_right, i_ext) > 0:
            v_right += 1
        while v_right - v_left > 1e-10:
            v_c = (v_left + v_right) / 2
            if f(v_c, i_ext) >= 0:
                v_left = v_c
            else:
                v_right = v_c
        return (v_left + v_right) / 2

    def jacobian(v_c):
        return np.array([[1 - v_c ** 2, -a], [a / tau_n, -1 / tau_n]])

    branches = {'red': [], 'green': [], 'black': [], 'blue': [], 'magenta': []}
    real_part_old = -1000.0
    i_c = None
    for ijk, i_ext in enumerate(i_ext_vec):
        v_c = find_fixed_point(i_ext)
        e = np.linalg.eigvals(jacobian(v_c))

        real_part = e[0].real
        if real_part > 0 and real_part_old < 0:
            i_c = (i_ext_vec[ijk] * (-real_part_old) + i_ext_vec[ijk - 1] * real_part) \
                / (real_part - real_part_old)
        real_part_old = real_part

        if abs(e[0].imag) > 1e-4:
            if e[0].real < 0:
                branches['red'].append((i_ext, v_c))
            if e[0].real > 0:
                branches['green'].append((i_ext, v_c))
        else:
            if e[0].real < 0 and e[1].real < 0:
                branches['black'].append((i_ext, v_c))
            if e[0].real > 0 and e[1].real > 0:
                branches['blue'].append((i_ext, v_c))
            if e[0].real > 0 and e[1].real < 0:
                branches['magenta'].append((i_ext, v_c))

    dt05 = dt / 2
    m_steps = round(t_final / dt)
    half = m_steps // 2

    cycle_pts = []
    for i in i_scan:
        v, n = 3.0, 3.0
        vmax, vmin = -np.inf, np.inf
        for k in range(m_steps):
            v_inc = v - v ** 3 / 3 - n + i
            n_inc = (a * v - n) / tau_n
            v_tmp = v + dt05 * v_inc
            n_tmp = n + dt05 * n_inc
            v_inc = v_tmp - v_tmp ** 3 / 3 - n_tmp + i
            n_inc = (a * v_tmp - n_tmp) / tau_n
            v = v + dt * v_inc
            n = n + dt * n_inc
            if k >= half:
                if v > vmax:
                    vmax = v
                if v < vmin:
                    vmin = v
        if vmax - vmin > 0.8:
            cycle_pts.append((i, vmax, vmin))

    return branches, i_c, cycle_pts, i_ext_vec


def plot_fitzhugh_nagumo_macro(branches, i_c, cycle_pts, i_ext_vec):
    plt.figure(figsize=(7, 7))

    if branches['blue']:
        i_pts, v_pts = zip(*branches['blue'])
        plt.plot(i_pts, v_pts, '-.b', linewidth=4)
    if branches['green']:
        i_pts, v_pts = zip(*branches['green'])
        plt.plot(i_pts, v_pts, '--g', linewidth=4)
    if branches['red']:
        i_pts, v_pts = zip(*branches['red'])
        plt.plot(i_pts, v_pts, '.r', markersize=10)
    if branches['black']:
        i_pts, v_pts = zip(*branches['black'])
        plt.plot(i_pts, v_pts, '-k', linewidth=4)
    if branches['magenta']:
        i_pts, v_pts = zip(*branches['magenta'])
        plt.plot(i_pts, v_pts, '--m', linewidth=4)

    for i, vmax, vmin in cycle_pts:
        plt.plot(i, vmax, '.k')
        plt.plot(i, vmin, '.k')

    plt.xlabel('$I$')
    plt.ylabel(r'$v_\ast$')
    plt.xlim(i_ext_vec[0], i_ext_vec[-1])
    plt.ylim(-5.7, 4.7)
    plt.gca().set_box_aspect(1)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_fitzhugh_nagumo_macro(*simulate_fitzhugh_nagumo_macro())

## FitzHugh-Nagumo Bifurcation Diagram (Micro View)

Zooms into the exponentially narrow window of $I$ around the Hopf point
found above, tracing the unstable-spiral-bounded canard-cycle envelope.

In [ ]:
def simulate_fitzhugh_nagumo_micro(a=5.0, tau_n=60.0, i_c=-4.291556094395530,
                                    i_ext_vec=None, t_final=10000.0, dt=0.01):
    if i_ext_vec is None:
        i_ext_vec = 1.01 * i_c + np.arange(501) / 500 * (0.98 * i_c - 1.01 * i_c)

    def f(v, i_ext):
        return v - v ** 3 / 3 - a * v + i_ext

    def find_fixed_point(i_ext):
        v_left, v_right = -1.0, 1.0
        while f(v_left, i_ext) < 0:
            v_left -= 1
        while f(v_right, i_ext) > 0:
            v_right += 1
        while v_right - v_left > 1e-10:
            v_c = (v_left + v_right) / 2
            if f(v_c, i_ext) >= 0:
                v_left = v_c
            else:
                v_right = v_c
        return (v_left + v_right) / 2

    def jacobian(v_c):
        return np.array([[1 - v_c ** 2, -a], [a / tau_n, -1 / tau_n]])

    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tail_start = 4 * m_steps // 5 - 1  # matlab's v(4*m_steps/5 : m_steps+1) is 1-indexed

    def cycle_amplitude(v_c, n_c, i_ext):
        """plain-float Heun integration (no array storage) -- just the max/min
        of v over the tail 1/5 of a 10s run, starting just outside the
        unstable spiral, needed for the limit-cycle envelope below"""
        v, n = v_c, n_c * 1.05
        vmax, vmin = -np.inf, np.inf
        for k in range(m_steps):
            v_inc = v - v ** 3 / 3 - n + i_ext
            n_inc = (a * v - n) / tau_n
            v_tmp = v + dt05 * v_inc
            n_tmp = n + dt05 * n_inc
            v_inc = v_tmp - v_tmp ** 3 / 3 - n_tmp + i_ext
            n_inc = (a * v_tmp - n_tmp) / tau_n
            v = v + dt * v_inc
            n = n + dt * n_inc
            if k >= tail_start:
                if v > vmax:
                    vmax = v
                if v < vmin:
                    vmin = v
        return vmax, vmin

    branches = {'red': [], 'green': [], 'black': [], 'blue': [], 'magenta': []}
    cycle = []
    real_part_old = -1000.0
    i_c_found = None

    for ijk, i_ext in enumerate(i_ext_vec):
        v_c = find_fixed_point(i_ext)
        n_c = a * v_c
        e = np.linalg.eigvals(jacobian(v_c))

        real_part = e[0].real
        if real_part > 0 and real_part_old < 0:
            i_c_found = (i_ext_vec[ijk] * (-real_part_old) + i_ext_vec[ijk - 1] * real_part) \
                / (real_part - real_part_old)
        real_part_old = real_part

        if abs(e[0].imag) > 1e-4:
            if e[0].real < 0:
                branches['red'].append((i_ext, v_c))
            if e[0].real > 0:
                branches['green'].append((i_ext, v_c))
        else:
            if e[0].real < 0 and e[1].real < 0:
                branches['black'].append((i_ext, v_c))
            if e[0].real > 0 and e[1].real > 0:
                branches['blue'].append((i_ext, v_c))
            if e[0].real > 0 and e[1].real < 0:
                branches['magenta'].append((i_ext, v_c))

        if abs(e[0].imag) > 1e-4 and e[0].real > 0:
            vmax, vmin = cycle_amplitude(v_c, n_c, i_ext)
            cycle.append((i_ext, vmax, vmin))

    return branches, cycle, i_c_found, i_ext_vec


def plot_fitzhugh_nagumo_micro(branches, cycle, i_c_found, i_ext_vec):
    plt.figure(figsize=(7, 7))

    if branches['blue']:
        i_pts, v_pts = zip(*branches['blue'])
        plt.plot(i_pts, v_pts, '-.b', linewidth=4)
    if branches['green']:
        i_pts, v_pts = zip(*branches['green'])
        plt.plot(i_pts, v_pts, '--g', linewidth=4)
    if branches['red']:
        i_pts, v_pts = zip(*branches['red'])
        plt.plot(i_pts, v_pts, '.r', markersize=10)
    if branches['black']:
        i_pts, v_pts = zip(*branches['black'])
        plt.plot(i_pts, v_pts, '-k', linewidth=4)
    if branches['magenta']:
        i_pts, v_pts = zip(*branches['magenta'])
        plt.plot(i_pts, v_pts, '--m', linewidth=4)
    if cycle:
        i_pts, max_pts, min_pts = zip(*cycle)
        plt.plot(i_pts, max_pts, ':k', linewidth=2)
        plt.plot(i_pts, min_pts, ':k', linewidth=2)

    plt.xlabel('$I$')
    plt.ylabel(r'$v_\ast$')
    plt.xticks([-4.3, -4.26, -4.22])
    plt.xlim(i_ext_vec[0], i_ext_vec[-1])
    plt.ylim(-2.5, 2)
    plt.gca().set_box_aspect(1)
    plt.tight_layout()
    plt.show()

In [ ]:
# Slow (~minutes): a full trajectory for every i_ext where the fixed
# point is an unstable spiral, to trace the canard-cycle envelope.
plot_fitzhugh_nagumo_micro(*simulate_fitzhugh_nagumo_micro())

## Reduced HH Bifurcation Diagram, Canard Explosion

Same construction as the FitzHugh-Nagumo bifurcation diagrams above, but
for the reduced HH model ($m=m_\infty(v)$, $h=0.83-n$).

In [ ]:
def simulate_hh_reduced_bif_diag(c=1.0, g_k=36.0, g_na=120.0, g_l=0.3,
                                  v_k=-82.0, v_na=45.0, v_l=-59.0,
                                  i_ext_vec=None, t_final=300.0, dt=0.01):
    # the backward (unstable-cycle) integration below genuinely blows up for a
    # handful of i_ext values before the "if Bmax<200" check discards it --
    # same as matlab, which silently lets Inf/NaN propagate there too
    np.seterr(all='ignore')

    if i_ext_vec is None:
        i_ext_vec = 5 + np.arange(101) / 100 * (10 - 5)

    def alpha_m(v):
        with np.errstate(divide='ignore', invalid='ignore'):
            out = (v + 45) / 10.0 / (1 - np.exp(-(v + 45) / 10))
        return out if abs(v + 45) > 1e-8 else 1.0

    def alpha_m_p(v):
        num, den = (v + 45) / 10, 1 - np.exp(-(v + 45) / 10)
        num_p, den_p = 1 / 10, np.exp(-(v + 45) / 10) / 10
        return (den * num_p - num * den_p) / den ** 2

    def alpha_n(v):
        return 0.01 * (-60.0 - v) / (np.exp((-60 - v) / 10) - 1)

    def alpha_n_p(v):
        num, den = 0.01 * (-60.0 - v), np.exp((-60 - v) / 10) - 1
        num_p, den_p = -0.01, -(den + 1) * 0.1
        return (den * num_p - num * den_p) / den ** 2

    def beta_m(v):
        return 4 * np.exp(-(v + 70) / 18)

    def beta_m_p(v):
        return -(4 / 18) * np.exp(-(v + 70) / 18)

    def beta_n(v):
        return 0.125 * np.exp(-(v + 70) / 80)

    def beta_n_p(v):
        return -beta_n(v) / 80

    def m_inf(v):
        return alpha_m(v) / (alpha_m(v) + beta_m(v))

    def m_inf_p(v):
        num = (alpha_m(v) + beta_m(v)) * alpha_m_p(v)
        num = num - alpha_m(v) * (alpha_m_p(v) + beta_m_p(v))
        return num / (alpha_m(v) + beta_m(v)) ** 2

    def n_inf(v):
        return alpha_n(v) / (alpha_n(v) + beta_n(v))

    def f(v):
        """dv/dt of the reduced (m=m_inf(v), h=0.83-n_inf(v)) HH model, i_ext=0"""
        return (g_na * m_inf(v) ** 3 * (0.83 - n_inf(v)) * (v_na - v)
                + g_k * n_inf(v) ** 4 * (v_k - v) + g_l * (v_l - v))

    def find_fixed_point(i_ext):
        v_left, v_right = -100.0, 50.0
        while v_right - v_left > 1e-10:
            v_c = (v_left + v_right) / 2
            if (f(v_c) + i_ext) * (f(v_left) + i_ext) > 0:
                v_left = v_c
            else:
                v_right = v_c
        return (v_left + v_right) / 2

    def jacobian(v, n):
        j00 = (g_na * 3 * m_inf(v) ** 2 * m_inf_p(v) * (0.83 - n) * (v_na - v)
               - g_na * m_inf(v) ** 3 * (0.83 - n) - g_k * n ** 4 - g_l)
        j01 = -g_na * m_inf(v) ** 3 * (v_na - v) + 4 * g_k * n ** 3 * (v_k - v)
        j10 = alpha_n_p(v) * (1 - n) - beta_n_p(v) * n
        j11 = -alpha_n(v) - beta_n(v)
        return np.array([[j00, j01], [j10, j11]]) / c

    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tail_start = round(2 / 3 * m_steps)

    def envelope(v0, n0, i_ext, direction):
        """Heun integration (plain floats), returns max/min of v over the
        tail 1/3 of the run. direction=+1 forward (settle onto the stable
        cycle), direction=-1 backward (trace the unstable cycle out from
        just next to the stable fixed point)."""
        v, n = v0, n0
        vmax, vmin = -np.inf, np.inf
        for k in range(m_steps):
            if not np.isfinite(v):
                return np.inf, -np.inf
            m = m_inf(v)
            h = 0.83 - n
            v_inc = (g_na * m ** 3 * h * (v_na - v) + g_k * n ** 4 * (v_k - v)
                     + g_l * (v_l - v) + i_ext) / c
            n_inc = alpha_n(v) * (1 - n) - beta_n(v) * n
            v_tmp = v + direction * dt05 * v_inc
            n_tmp = n + direction * dt05 * n_inc
            m_tmp = m_inf(v_tmp)
            h_tmp = 0.83 - n_tmp
            v_inc = (g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp) + g_k * n_tmp ** 4 * (v_k - v_tmp)
                     + g_l * (v_l - v_tmp) + i_ext) / c
            n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp
            v = v + direction * dt * v_inc
            n = n + direction * dt * n_inc
            if k >= tail_start:
                if v > vmax:
                    vmax = v
                if v < vmin:
                    vmin = v
        return vmax, vmin

    green, red, uc, sc = [], [], [], []
    for i_ext in i_ext_vec:
        v_c = find_fixed_point(i_ext)
        n_c = n_inf(v_c)
        e = np.linalg.eigvals(jacobian(v_c, n_c))

        if e[0].real < 0 and e[1].real < 0 and abs(e[0].imag) > 1e-4:
            red.append((i_ext, v_c))
            Bmax, Bmin = envelope(v_c * 1.01, n_c * 0.99, i_ext, direction=-1)
            if Bmax < 200:
                uc.append((i_ext, Bmax, Bmin))

        if e[0].real > 0 and e[1].real > 0 and abs(e[0].imag) > 1e-4:
            green.append((i_ext, v_c))

        Amax, Amin = envelope(70.0, 0.5, i_ext, direction=1)
        if Amax - Amin > 2:
            sc.append((i_ext, Amax, Amin))

    return red, green, uc, sc


def plot_hh_reduced_bif_diag(red, green, uc, sc):
    plt.figure(figsize=(7, 7))

    if green:
        i_pts, v_pts = zip(*green)
        plt.plot(i_pts, v_pts, '--g', linewidth=4)
    if red:
        i_pts, v_pts = zip(*red)
        plt.plot(i_pts, v_pts, '-r', linewidth=4)

    i_sc, amax, amin = zip(*sc)
    plt.plot(i_sc, amax, '-k', linewidth=1)
    plt.plot(i_sc, amin, '-k', linewidth=1)

    i_uc, bmax, bmin = zip(*uc)
    i_uc = [i_sc[0], *i_uc]
    bmax = [amax[0], *bmax]
    bmin = [amin[0], *bmin]
    plt.plot(i_uc, bmax, '--k', linewidth=2)
    plt.plot(i_uc, bmin, '--k', linewidth=2)

    plt.xticks([])
    plt.ylabel('$v$')
    plt.xlim(5, 10)
    plt.ylim(-90, 50)
    plt.text(4.5, -100, r'$I_\ast \approx 5.25$', fontsize=14)
    plt.text(7.4 - 0.75, -100, r'$I_c \approx 7.4$', fontsize=14)
    plt.plot([5.25, 5.25], [-92, -87], '-k', linewidth=1)
    plt.plot([7.4, 7.4], [-92, -87], '-k', linewidth=1)
    plt.gca().set_box_aspect(1)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hh_reduced_bif_diag(*simulate_hh_reduced_bif_diag())

## Mixed-Mode Oscillations

Adds spike-triggered adaptation ($I_{adapt}$ jumps by $-\delta$ each time
$v$ crosses zero downward, then decays with time constant
$\tau_{adapt}$) to the FitzHugh-Nagumo model, producing an irregular
pattern of small subthreshold wiggles and large spikes.

In [ ]:
def simulate_mmos(a=5.0, tau_n=60.0, i_ext=-4.2, tau_adapt=150.0, delta=0.2,
                   t_final=1000.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    v = np.zeros(m_steps + 1)
    n = np.zeros(m_steps + 1)
    v[0], n[0] = -1.0, -4.75
    i_adapt = -delta

    for k in range(m_steps):
        v_inc = v[k] - v[k] ** 3 / 3 - n[k] + i_ext + i_adapt
        n_inc = (a * v[k] - n[k]) / tau_n
        v_tmp = v[k] + dt05 * v_inc
        n_tmp = n[k] + dt05 * n_inc
        v_inc = v_tmp - v_tmp ** 3 / 3 - n_tmp + i_ext + i_adapt * np.exp(-dt05 / tau_adapt)
        n_inc = (a * v_tmp - n_tmp) / tau_n
        v[k + 1] = v[k] + dt * v_inc
        n[k + 1] = n[k] + dt * n_inc
        i_adapt = i_adapt * np.exp(-dt / tau_adapt)
        if v[k + 1] < 0 and v[k] >= 0:
            i_adapt = i_adapt - delta

    t = np.arange(m_steps + 1) * dt
    return t, v


def plot_mmos(t, v):
    plt.figure(figsize=(7, 3.5))
    plt.plot(t, v, '-k', linewidth=2)
    plt.xlim(0, t[-1])
    plt.ylim(-3, 3)
    plt.xlabel('$t$')
    plt.ylabel('$v$')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_mmos(*simulate_mmos())

In [ ]:
interact(lambda i_ext=-4.2, delta=0.2: plot_mmos(*simulate_mmos(i_ext=i_ext, delta=delta)),
         i_ext=(-4.5, -3.8, 0.05), delta=(0.0, 0.5, 0.02));